In [0]:
users_df = spark.read.csv("/Volumes/hexa_databricks_7405605275266839/default/datasets/users.csv", header=True, inferSchema=True)
expenses_df = spark.read.csv("/Volumes/hexa_databricks_7405605275266839/default/datasets/expenses.csv", header=True, inferSchema=True)

In [0]:
users_df.printSchema()
expenses_df.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- user_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- income: integer (nullable = true)

root
 |-- user_id: integer (nullable = true)
 |-- month: date (nullable = true)
 |-- category: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- payment_mode: string (nullable = true)



In [0]:
display(users_df)

user_id,user_name,city,age,income
1,Arun,Chennai,32,18000
2,Priya,Bangalore,34,15000
3,Karthik,Hyderabad,24,25000
4,Divya,Mumbai,25,20000
5,Ramesh,Delhi,40,15000
6,Sneha,Pune,38,18000


In [0]:
display(expenses_df)

user_id,month,category,amount,payment_mode
5,2025-02-01,Entertainment,988,Card
1,2025-01-01,Food,4052,Cash
4,2025-02-01,Food,5264,Card
3,2025-02-01,Shopping,3933,UPI
6,2025-02-01,Entertainment,2535,UPI
3,2025-01-01,Travel,881,Card
2,2025-01-01,Food,2328,Card
4,2025-01-01,Entertainment,3027,Card
2,2025-02-01,Entertainment,5296,Cash
5,2025-01-01,Shopping,1298,Card


In [0]:
combined_df = expenses_df.join(
    users_df, 
    "user_id", 
    "left")
    
combined_df.show()

+-------+----------+-------------+------+------------+---------+---------+---+------+
|user_id|     month|     category|amount|payment_mode|user_name|     city|age|income|
+-------+----------+-------------+------+------------+---------+---------+---+------+
|      5|2025-02-01|Entertainment|   988|        Card|   Ramesh|    Delhi| 40| 15000|
|      1|2025-01-01|         Food|  4052|        Cash|     Arun|  Chennai| 32| 18000|
|      4|2025-02-01|         Food|  5264|        Card|    Divya|   Mumbai| 25| 20000|
|      3|2025-02-01|     Shopping|  3933|         UPI|  Karthik|Hyderabad| 24| 25000|
|      6|2025-02-01|Entertainment|  2535|         UPI|    Sneha|     Pune| 38| 18000|
|      3|2025-01-01|       Travel|   881|        Card|  Karthik|Hyderabad| 24| 25000|
|      2|2025-01-01|         Food|  2328|        Card|    Priya|Bangalore| 34| 15000|
|      4|2025-01-01|Entertainment|  3027|        Card|    Divya|   Mumbai| 25| 20000|
|      2|2025-02-01|Entertainment|  5296|        Cash|

In [0]:
from pyspark.sql.functions import *

monthly_spend_df = combined_df.groupBy("user_id", "user_name", "city", "month").agg(
    sum("amount").alias("monthly_spend")
)
monthly_spend_df.show()

+-------+---------+---------+----------+-------------+
|user_id|user_name|     city|     month|monthly_spend|
+-------+---------+---------+----------+-------------+
|      6|    Sneha|     Pune|2025-02-01|         5997|
|      3|  Karthik|Hyderabad|2025-02-01|         3933|
|      3|  Karthik|Hyderabad|2025-01-01|          881|
|      5|   Ramesh|    Delhi|2025-01-01|         1298|
|      1|     Arun|  Chennai|2025-01-01|         4052|
|      4|    Divya|   Mumbai|2025-02-01|         5264|
|      1|     Arun|  Chennai|2025-02-01|         1243|
|      6|    Sneha|     Pune|2025-01-01|         4855|
|      2|    Priya|Bangalore|2025-02-01|         5296|
|      5|   Ramesh|    Delhi|2025-02-01|          988|
|      2|    Priya|Bangalore|2025-01-01|         7967|
|      4|    Divya|   Mumbai|2025-01-01|         3027|
+-------+---------+---------+----------+-------------+



In [0]:
summary_df = monthly_spend_df.join(users_df.select("user_id", "income"), "user_id", "left")

display(summary_df)

user_id,user_name,city,month,monthly_spend,income
6,Sneha,Pune,2025-02-01,5997,18000
3,Karthik,Hyderabad,2025-02-01,3933,25000
3,Karthik,Hyderabad,2025-01-01,881,25000
5,Ramesh,Delhi,2025-01-01,1298,15000
1,Arun,Chennai,2025-01-01,4052,18000
4,Divya,Mumbai,2025-02-01,5264,20000
1,Arun,Chennai,2025-02-01,1243,18000
6,Sneha,Pune,2025-01-01,4855,18000
2,Priya,Bangalore,2025-02-01,5296,15000
5,Ramesh,Delhi,2025-02-01,988,15000


In [0]:
summary_df = summary_df.withColumn("savings", col("income") - col("monthly_spend"))
summary_df.show()

+-------+---------+---------+----------+-------------+------+-------+
|user_id|user_name|     city|     month|monthly_spend|income|savings|
+-------+---------+---------+----------+-------------+------+-------+
|      6|    Sneha|     Pune|2025-02-01|         5997| 18000|  12003|
|      3|  Karthik|Hyderabad|2025-02-01|         3933| 25000|  21067|
|      3|  Karthik|Hyderabad|2025-01-01|          881| 25000|  24119|
|      5|   Ramesh|    Delhi|2025-01-01|         1298| 15000|  13702|
|      1|     Arun|  Chennai|2025-01-01|         4052| 18000|  13948|
|      4|    Divya|   Mumbai|2025-02-01|         5264| 20000|  14736|
|      1|     Arun|  Chennai|2025-02-01|         1243| 18000|  16757|
|      6|    Sneha|     Pune|2025-01-01|         4855| 18000|  13145|
|      2|    Priya|Bangalore|2025-02-01|         5296| 15000|   9704|
|      5|   Ramesh|    Delhi|2025-02-01|          988| 15000|  14012|
|      2|    Priya|Bangalore|2025-01-01|         7967| 15000|   7033|
|      4|    Divya| 

In [0]:
final_df = summary_df.withColumn(
    "alert",
    when(col("monthly_spend") > 15000, "High Spending").otherwise("Normal")
)
final_df.show()

+-------+---------+---------+----------+-------------+------+-------+------+
|user_id|user_name|     city|     month|monthly_spend|income|savings| alert|
+-------+---------+---------+----------+-------------+------+-------+------+
|      6|    Sneha|     Pune|2025-02-01|         5997| 18000|  12003|Normal|
|      3|  Karthik|Hyderabad|2025-02-01|         3933| 25000|  21067|Normal|
|      3|  Karthik|Hyderabad|2025-01-01|          881| 25000|  24119|Normal|
|      5|   Ramesh|    Delhi|2025-01-01|         1298| 15000|  13702|Normal|
|      1|     Arun|  Chennai|2025-01-01|         4052| 18000|  13948|Normal|
|      4|    Divya|   Mumbai|2025-02-01|         5264| 20000|  14736|Normal|
|      1|     Arun|  Chennai|2025-02-01|         1243| 18000|  16757|Normal|
|      6|    Sneha|     Pune|2025-01-01|         4855| 18000|  13145|Normal|
|      2|    Priya|Bangalore|2025-02-01|         5296| 15000|   9704|Normal|
|      5|   Ramesh|    Delhi|2025-02-01|          988| 15000|  14012|Normal|

In [0]:
payment_count_df = combined_df.groupBy("user_id", "payment_mode").count()
payment_count_df.show()

+-------+------------+-----+
|user_id|payment_mode|count|
+-------+------------+-----+
|      2|        Cash|    1|
|      1|        Card|    1|
|      3|         UPI|    1|
|      6|         UPI|    1|
|      5|        Card|    2|
|      1|        Cash|    1|
|      6|        Cash|    2|
|      3|        Card|    1|
|      2|        Card|    2|
|      4|        Card|    2|
+-------+------------+-----+



In [0]:
final_df.write.format("delta").mode("overwrite").save("/Volumes/hexa_databricks_7405605275266839/default/datasets/expense_summary_delta")

final_df.write.format("csv").mode("overwrite").option("header", True).save("/Volumes/hexa_databricks_7405605275266839/default/datasets/expense_summary_csv")

print("ETL done! Summary table saved.")

ETL done! Summary table saved.


In [0]:
join_df = expenses_df.join(users_df, on="user_id", how="left")

users_tot = join_df.groupBy("user_id", "user_name").agg(
    sum("amount").alias("total_spent")
)

print("Report: ")
print("\nTotal users: ", users_df.count())
print("Total expenses: ", expenses_df.count())

print("\nUser with most expenses: ")
users_tot.orderBy(col("total_spent").desc()).show(1)

print("\nUser with least expenses: ")
users_tot.orderBy(col("total_spent").asc()).show(1)

print("\nMost expensive expense: ")
join_df.orderBy(col("amount").desc()).show(1)

print("\nLeast expensive expense: ")
join_df.orderBy(col("amount").asc()).show(1)

print("\nTotal expenses by category: ")
join_df.groupBy("category").sum("amount").show()

print("\nTotal expenses by user: ")
users_tot.show()

Report: 

Total users:  6
Total expenses:  14

User with most expenses: 
+-------+---------+-----------+
|user_id|user_name|total_spent|
+-------+---------+-----------+
|      2|    Priya|      13263|
+-------+---------+-----------+
only showing top 1 row

User with least expenses: 
+-------+---------+-----------+
|user_id|user_name|total_spent|
+-------+---------+-----------+
|      5|   Ramesh|       2286|
+-------+---------+-----------+
only showing top 1 row

Most expensive expense: 
+-------+----------+-------------+------+------------+---------+---------+---+------+
|user_id|     month|     category|amount|payment_mode|user_name|     city|age|income|
+-------+----------+-------------+------+------------+---------+---------+---+------+
|      2|2025-01-01|Entertainment|  5639|        Card|    Priya|Bangalore| 34| 15000|
+-------+----------+-------------+------+------------+---------+---------+---+------+
only showing top 1 row

Least expensive expense: 
+-------+----------+-------